In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
# ------------

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-14 22:35:38--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2025-11-14 22:35:39 (14.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
torch.manual_seed(1337)

#
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [8]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long).to(device)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [10]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x, y
    return x, y

In [11]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [13]:
tabela_probabilidades = nn.Embedding(vocab_size, vocab_size)


In [14]:
tabela_probabilidades(torch.tensor([18]))

tensor([[ 1.0055, -0.0839,  0.5748,  0.5065, -0.3243,  1.1693, -0.2089, -0.4413,
         -0.7053, -0.7238, -1.6831, -0.2750, -0.7649, -0.6673,  1.3645, -2.5823,
         -0.0642, -1.2065,  0.5845, -0.2359,  1.1295, -0.8724,  0.5468, -0.5030,
         -1.1575, -1.2978, -0.3060,  0.4138,  0.6090, -1.7799, -1.3177, -0.6032,
         -0.6397, -0.8189, -0.8130,  0.3439,  0.9922,  0.8235,  0.6157,  1.4101,
          0.3128,  0.6790, -1.4544,  0.0282, -1.0967,  0.7178, -0.2200,  0.0422,
         -0.1580,  1.1402, -1.5646, -0.8132, -0.9053, -0.7255,  0.2575, -0.0661,
          0.1208, -2.7387, -0.3296, -1.1953,  1.4944,  0.2142, -0.6021,  0.2213,
          1.3709]], grad_fn=<EmbeddingBackward0>)

In [15]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [17]:
model = BigramLanguageModel(vocab_size).to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.6032, val loss 4.6301
step 300: train loss 2.7979, val loss 2.8258
step 600: train loss 2.5412, val loss 2.5774
step 900: train loss 2.4945, val loss 2.5231
step 1200: train loss 2.4727, val loss 2.5035
step 1500: train loss 2.4650, val loss 2.4959
step 1800: train loss 2.4708, val loss 2.4980
step 2100: train loss 2.4665, val loss 2.4881
step 2400: train loss 2.4582, val loss 2.4907
step 2700: train loss 2.4683, val loss 2.4976


In [18]:

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))



Core mewime wint:
Pulben beresis co ck to uses fofor met ckin ghithopan me;PESThak o asppll is?

KI chifit,

Mofrede aWhorowin th's y ndatcaranginerr, s tZWe an'seeles y al E:
RWere.
In findispon KI pererousullicl th ino or stherpominof allo y he blldsunystoffos!
DYOMA yoyeshervece ll oy or, Y f te
CEd whekspHEdeatr'Tomam; f as.
I he I p'st omy, ad d f frm an gincallenger wesemy whownit thout bentiakesherousur in plos t wns d arofulaldoom
warerevecese yout s tow, d
Toncheea;'s nelll fatingrt a h


In [ ]:
# Generate 500 characters random text
rand = torch.randint(0, vocab_size, (500,))
print(decode(rand.tolist()))

 t pvpHSd3B;NR;r;nmm-&.TG3WQ.,?PiglakDoSwWLrvLgNDcdIBq&?SXmytm,yXogILRZa3FcJt!HJvlXmUgavwVBcGsAe:hh'hpvdFyTec.:vpHfkrqpYyeO!m
Wo;u$J-:HnmZx!oI'Bvg,F-,Tqj-UFW CtnW?fcGaV LetPFe;LsxFEgfqruxmPQF'zVxIfIw?KgZ:'RXS3RLwHKVjvkPT?RbAuadgQcRsayNM,a&tl.VSZPJlESuhTgDMZJEfFKWqEHBRwPe.HGdLpt:LlzVX$Yl;Rar;E!maiex3mjaNcNT!fj?mONZ.KOJMzhMc:AVRPIJlUezXqm3;yng&affF-uV:hNnDmjNzr3tUM?L? W3wJckueZspIDIrN.;;agYr?yKeZqmxlPPshSD GiDbTZ
OmKbd,j!W tPFedRks$ESqMwB.s;AUUScPSF.fQN?TEBh,NfpjeiWIkkw&'B:QrAJNe!CGXsJih
.JtF3?-d?
